# Tiền xử lý dữ liệu — House Price Prediction



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE    = 0.2

ROOT = Path.cwd()
if not (ROOT / 'train.csv').exists():
    ROOT = ROOT.parent

train = pd.read_csv(ROOT / 'train.csv')
print('Train:', train.shape)

Train: (1460, 81)


## 1. Làm sạch & Feature Engineering

In [2]:

NONE_COLUMNS = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
    'BsmtFinType2', 'MasVnrType',
]
ZERO_COLUMNS = [
    'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2',
    'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
    'MasVnrArea',
]
QUALITY_COLUMNS = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC',
]
QUALITY_MAP = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
OUTLIER_GR_LIV_AREA = 4000
OUTLIER_PRICE        = 300_000

def clean_and_engineer(frame: pd.DataFrame, is_train: bool = False) -> pd.DataFrame:
    result = frame.copy()
    if is_train and 'SalePrice' in result.columns:
        result = result.loc[
            ~((result['GrLivArea'] > OUTLIER_GR_LIV_AREA)
              & (result['SalePrice'] < OUTLIER_PRICE))
        ].copy()

    result = result.drop(columns=['Id', 'GarageYrBlt'], errors='ignore')

    for col in NONE_COLUMNS:
        if col in result.columns:
            result[col] = result[col].fillna('None')
    for col in ZERO_COLUMNS:
        if col in result.columns:
            result[col] = result[col].fillna(0)

    result['TotalSF']   = result['TotalBsmtSF'] + result['1stFlrSF'] + result['2ndFlrSF']
    result['HouseAge']  = (result['YrSold'] - result['YearBuilt']).clip(lower=0)
    result['RemodAge']  = (result['YrSold'] - result['YearRemodAdd']).clip(lower=0)
    result['TotalBath'] = (
        result['FullBath']
        + 0.5 * result['HalfBath']
        + result['BsmtFullBath']
        + 0.5 * result['BsmtHalfBath']
    )
    result['MSSubClass'] = result['MSSubClass'].astype(str)

    for col in QUALITY_COLUMNS:
        if col in result.columns:
            result[col] = result[col].map(QUALITY_MAP)

    return result

train = clean_and_engineer(train, is_train=True)

X = train.drop(columns='SalePrice')
y = np.log1p(train['SalePrice']).rename('log_SalePrice')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f'Sau làm sạch — X_train: {X_train.shape} | X_test: {X_test.shape}')

Sau làm sạch — X_train: (1166, 82) | X_test: (292, 82)


## 2. Pipeline tiền xử lý (Impute → Log/Scale → OHE)

In [3]:

group_medians = X_train.groupby('Neighborhood')['LotFrontage'].median()
global_median = X_train['LotFrontage'].median()

for frame in (X_train, X_test):
    frame['LotFrontage'] = (
        frame['LotFrontage']
        .fillna(frame['Neighborhood'].map(group_medians))
        .fillna(global_median)
    )

num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(exclude='number').columns.tolist()
skewed   = X_train[num_cols].skew().abs().gt(0.75)
skew_cols   = skewed[skewed].index.tolist()
normal_cols = [c for c in num_cols if c not in skew_cols]

print(f'Số cột số lệch (skew > 0.75): {len(skew_cols)}')
print(f'Số cột số bình thường       : {len(normal_cols)}')
print(f'Số cột phân loại            : {len(cat_cols)}')

skew_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('log',   FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scale', StandardScaler()),
])
normal_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', min_frequency=2, sparse_output=False)),
])

preprocessor = ColumnTransformer(
    [
        ('skew',   skew_pipe,   skew_cols),
        ('normal', normal_pipe, normal_cols),
        ('cat',    cat_pipe,    cat_cols),
    ],
    verbose_feature_names_out=False,
)

feature_names = None
X_train_proc = pd.DataFrame(
    preprocessor.fit_transform(X_train),
    columns=preprocessor.get_feature_names_out(),
)
X_test_proc = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=preprocessor.get_feature_names_out(),
)
print(f'X_train_proc: {X_train_proc.shape} | X_test_proc: {X_test_proc.shape}')

Số cột số lệch (skew > 0.75): 27
Số cột số bình thường       : 21
Số cột phân loại            : 34
X_train_proc: (1166, 265) | X_test_proc: (292, 265)


## 3. Lưu dữ liệu đã xử lý

In [4]:
train_part = X_train_proc.copy()
train_part['log_SalePrice'] = y_train.to_numpy()
train_part['split'] = 'train'

test_part = X_test_proc.copy()
test_part['log_SalePrice'] = y_test.to_numpy()
test_part['split'] = 'test'

output = pd.concat([train_part, test_part], ignore_index=True)

assert not output.drop(columns='split').isnull().any().any(), 'Còn giá trị NaN!'

output.to_csv(ROOT / 'train_processed.csv', index=False)
print('✅ Đã lưu:', output.shape, '->', ROOT / 'train_processed.csv')
output.head(3)

✅ Đã lưu: (1458, 267) -> c:\Users\ADMIN\Documents\Machine-Learning\Tuan04\train_processed.csv


,LotFrontage,LotArea,MasVnrArea,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,...,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,log_SalePrice,split
0,0.150907,-0.159702,-0.804837,-0.660348,2.464599,-0.286692,0.157151,0.859124,-0.367925,0.194807,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,11.884496,train
1,-0.382255,-0.295247,-0.804837,1.114398,-0.189873,0.492939,0.157151,-1.435115,-0.367925,0.571664,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,12.089544,train
2,0.281985,-0.079944,0.857393,-0.660348,-0.189873,-0.286692,0.157151,0.823032,-0.367925,-0.234125,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,11.908347,train
